In [0]:
# =============================================================
# dq_profiler.py
# Layer   : Audit
# Purpose : Standalone validation for the eight fact tables.
#
# Reads:
#   Bronze Delta and Silver Delta
#
# Writes:
#   Nothing directly. The runner calls audit_writer.py.
#
# This notebook deliberately does not import or modify the working
# Bronze/Silver/Gold pipeline logic.
# =============================================================

from pyspark.sql import functions as F


def _has_col(df, column_name: str) -> bool:
    return df is not None and column_name in df.columns


def _count_where(df, condition) -> int:
    if df is None:
        return 0
    try:
        return df.filter(condition).count()
    except Exception as exc:
        print(f"[dq_profiler] Count failed: {exc}")
        return -1


def _try_read_delta(path: str):
    try:
        return spark.read.format("delta").load(path), ""
    except Exception as exc:
        return None, str(exc)[:500]


def _null_key_count(df, pk_cols: list[str]) -> int:
    if df is None:
        return 0
    missing = [c for c in pk_cols if c not in df.columns]
    if missing:
        return -1
    condition = None
    for c in pk_cols:
        next_cond = F.col(c).isNull() | (F.trim(F.col(c).cast("string")) == "")
        condition = next_cond if condition is None else (condition | next_cond)
    return _count_where(df, condition)


def _duplicate_key_count(df, pk_cols: list[str]) -> int:
    if df is None:
        return 0
    missing = [c for c in pk_cols if c not in df.columns]
    if missing:
        return -1
    total = df.count()
    distinct_count = df.select(*pk_cols).distinct().count()
    return max(total - distinct_count, 0)


def _schema_mismatch_details(df, table_name: str) -> tuple[int, str]:
    if df is None:
        return 1, "silver dataframe missing"

    expected = EXPECTED_SILVER_TYPES.get(table_name, {})
    mismatches = []
    actual_types = {field.name: field.dataType.simpleString().lower() for field in df.schema.fields}

    for col_name, expected_type in expected.items():
        actual = actual_types.get(col_name)
        if actual is None:
            mismatches.append(f"{col_name}: missing expected {expected_type}")
            continue

        allowed = [expected_type]
        if expected_type == "int":
            allowed = ["int", "integer"]
        if expected_type == "double":
            allowed = ["double", "float"]

        if actual not in allowed:
            mismatches.append(f"{col_name}: expected {expected_type}, actual {actual}")

    return len(mismatches), "; ".join(mismatches)


def _rule(run_id: str, table_name: str, load_date: str, rule_id: str, rule_name: str,
          layer: str, severity: str, measured_count: int, total_count: int,
          threshold: int, description: str, recommendation: str) -> dict:
    if severity == "INFO":
        status = "OBSERVED" if measured_count > threshold else "PASS"
    elif measured_count < 0:
        status = "ERROR"
    elif measured_count > threshold:
        status = "FAIL" if severity == "ERROR" else "WARN"
    else:
        status = "PASS"

    pct = 0.0
    if total_count and total_count > 0 and measured_count >= 0:
        pct = round((measured_count / total_count) * 100, 4)

    return {
        "run_id": run_id,
        "table_name": table_name,
        "load_date": normalize_date(load_date),
        "rule_id": rule_id,
        "rule_name": rule_name,
        "layer": layer,
        "severity": severity,
        "status": status,
        "measured_count": int(measured_count),
        "total_count": int(total_count or 0),
        "measured_pct": float(pct),
        "threshold": int(threshold),
        "description": description,
        "recommendation": recommendation,
    }


def _add_general_rules(rules: list[dict], run_id: str, table_name: str, load_date: str,
                       bronze_df, silver_df, bronze_rows: int, silver_rows_for_date: int,
                       bronze_error: str, silver_error: str) -> None:
    pk_cols = PRIMARY_KEYS[table_name]

    rules.append(_rule(
        run_id, table_name, load_date, "GENERAL_001", "Bronze Delta table exists",
        "bronze", "ERROR", 0 if bronze_df is not None else 1, 1, 0,
        "Bronze must exist before validation can compare Raw-to-Silver behavior.",
        bronze_error or "Check Bronze notebook execution and ADLS path."
    ))

    rules.append(_rule(
        run_id, table_name, load_date, "GENERAL_002", "Silver Delta table exists",
        "silver", "ERROR", 0 if silver_df is not None else 1, 1, 0,
        "Silver must exist because validation checks unresolved anomalies after cleaning.",
        silver_error or "Run nb_bronze_to_silver before audit validation."
    ))

    rules.append(_rule(
        run_id, table_name, load_date, "GENERAL_003", "Bronze rows for load date",
        "bronze", "WARN", 0 if bronze_rows > 0 else 1, 1, 0,
        "The selected load_date should have Bronze rows for a meaningful audit.",
        "If this warns, verify ADF window_start_time and Bronze _load_date values."
    ))

    rules.append(_rule(
        run_id, table_name, load_date, "GENERAL_004", "Silver rows for business date",
        "silver", "WARN", 0 if silver_rows_for_date > 0 else 1, 1, 0,
        "Silver should have rows for the selected business date after cleaning.",
        "If this warns, run Silver for this date or inspect the date column mapping."
    ))

    bronze_null_pk = _null_key_count(bronze_df, pk_cols)
    silver_null_pk = _null_key_count(silver_df, pk_cols)
    bronze_dup_pk = _duplicate_key_count(bronze_df, pk_cols)
    silver_dup_pk = _duplicate_key_count(silver_df, pk_cols)

    rules.append(_rule(
        run_id, table_name, load_date, "PK_001", "Bronze null or blank primary keys",
        "bronze", "WARN", bronze_null_pk, bronze_rows, 0,
        "Null business keys in Bronze indicate bad source records or generator issues.",
        "Trace source files using Bronze lineage and decide whether to quarantine/drop."
    ))
    rules.append(_rule(
        run_id, table_name, load_date, "PK_002", "Silver null or blank primary keys",
        "silver", "ERROR", silver_null_pk, silver_rows_for_date, 0,
        "Silver should not contain null primary keys after cleaning.",
        "Fix cleaner rules or enforce a quarantine/drop rule before Silver write."
    ))
    rules.append(_rule(
        run_id, table_name, load_date, "PK_003", "Bronze duplicate primary keys",
        "bronze", "INFO", bronze_dup_pk, bronze_rows, 0,
        "Duplicate keys in Bronze are expected evidence for deduplication checks.",
        "Verify Silver resolves duplicates where the table grain requires uniqueness."
    ))
    rules.append(_rule(
        run_id, table_name, load_date, "PK_004", "Silver duplicate primary keys",
        "silver", "ERROR", silver_dup_pk, silver_rows_for_date, 0,
        "Silver should be unique at its table grain.",
        "Review dropDuplicates/MERGE primary key logic in silver_writer.py."
    ))

    mismatch_count, details = _schema_mismatch_details(silver_df, table_name)
    rules.append(_rule(
        run_id, table_name, load_date, "SCHEMA_001", "Silver type contract",
        "silver", "ERROR", mismatch_count, len(EXPECTED_SILVER_TYPES.get(table_name, {})), 0,
        "Silver must expose clean typed columns for downstream Gold and Power BI.",
        details or "Schema matches expected Silver contract."
    ))


def _add_table_specific_rules(rules: list[dict], run_id: str, table_name: str,
                              load_date: str, bronze_df, silver_df,
                              bronze_rows: int, silver_rows_for_date: int) -> None:
    # Bronze anomaly rules are informational: dirty raw data is the point of the demo.
    # Silver unresolved rules are errors: Silver should have fixed them.
    if table_name == "sales":
        rules.append(_rule(run_id, table_name, load_date, "BR_SALES_001", "Bronze negative sale_price", "bronze", "INFO",
                           _count_where(bronze_df, F.expr("try_cast(sale_price AS double)") < 0), bronze_rows, 0,
                           "Incoming sales can contain invalid negative prices.", "Silver should nullify negative sale_price."))
        rules.append(_rule(run_id, table_name, load_date, "BR_SALES_002", "Bronze discount greater than sale_price", "bronze", "INFO",
                           _count_where(bronze_df, F.expr("try_cast(discount_amount AS double)") > F.expr("try_cast(sale_price AS double)")), bronze_rows, 0,
                           "Discounts larger than sale price distort revenue.", "Silver should cap discount_amount at sale_price."))
        rules.append(_rule(run_id, table_name, load_date, "SV_SALES_001", "Silver negative sale_price unresolved", "silver", "ERROR",
                           _count_where(silver_df, F.col("sale_price") < 0), silver_rows_for_date, 0,
                           "Negative revenue values should not survive Silver.", "Check _clean_sales sale_price rule."))
        rules.append(_rule(run_id, table_name, load_date, "SV_SALES_002", "Silver discount greater than sale_price unresolved", "silver", "ERROR",
                           _count_where(silver_df, F.col("discount_amount") > F.col("sale_price")), silver_rows_for_date, 0,
                           "Discount should not exceed sale price.", "Check _clean_sales discount capping rule."))
        rules.append(_rule(run_id, table_name, load_date, "SV_SALES_003", "Silver blank sales_channel unresolved", "silver", "ERROR",
                           _count_where(silver_df, F.col("sales_channel").isNull() | (F.trim(F.col("sales_channel")) == "")), silver_rows_for_date, 0,
                           "Blank sales channels break channel reporting.", "Check _clean_sales Unknown fallback rule."))

    elif table_name == "device_snapshots":
        rules.append(_rule(run_id, table_name, load_date, "BR_DEVICE_001", "Bronze battery out of range", "bronze", "INFO",
                           _count_where(bronze_df, (F.expr("try_cast(battery_pct AS double)") < 0) | (F.expr("try_cast(battery_pct AS double)") > 100)), bronze_rows, 0,
                           "Battery percentage should be between 0 and 100.", "Silver should nullify out-of-range battery_pct."))
        rules.append(_rule(run_id, table_name, load_date, "BR_DEVICE_002", "Bronze temperature sensor dropout", "bronze", "INFO",
                           _count_where(bronze_df, F.expr("try_cast(temperature_c AS double)") == 0.0), bronze_rows, 0,
                           "A temperature of 0.0 represents sensor dropout in this project.", "Silver should nullify temperature_c = 0.0."))
        rules.append(_rule(run_id, table_name, load_date, "SV_DEVICE_001", "Silver battery out of range unresolved", "silver", "ERROR",
                           _count_where(silver_df, (F.col("battery_pct") < 0) | (F.col("battery_pct") > 100)), silver_rows_for_date, 0,
                           "Invalid battery values should not survive Silver.", "Check _clean_device_snapshots battery rule."))
        rules.append(_rule(run_id, table_name, load_date, "SV_DEVICE_002", "Silver temperature dropout unresolved", "silver", "ERROR",
                           _count_where(silver_df, F.col("temperature_c") == 0.0), silver_rows_for_date, 0,
                           "Sensor dropout should not be treated as real temperature.", "Check _clean_device_snapshots temperature rule."))

    elif table_name == "activity_snapshots":
        rules.append(_rule(run_id, table_name, load_date, "BR_ACTIVITY_001", "Bronze negative steps", "bronze", "INFO",
                           _count_where(bronze_df, F.expr("try_cast(steps AS int)") < 0), bronze_rows, 0,
                           "Negative steps are invalid activity telemetry.", "Silver should convert negative steps to absolute values."))
        rules.append(_rule(run_id, table_name, load_date, "BR_ACTIVITY_002", "Bronze active_minutes over hourly grain", "bronze", "INFO",
                           _count_where(bronze_df, F.expr("try_cast(active_minutes AS int)") > 60), bronze_rows, 0,
                           "Hourly active minutes cannot exceed 60.", "Silver should cap active_minutes at 60."))
        rules.append(_rule(run_id, table_name, load_date, "SV_ACTIVITY_001", "Silver negative steps unresolved", "silver", "ERROR",
                           _count_where(silver_df, F.col("steps") < 0), silver_rows_for_date, 0,
                           "Negative steps should not survive Silver.", "Check _clean_activity_snapshots steps rule."))
        rules.append(_rule(run_id, table_name, load_date, "SV_ACTIVITY_002", "Silver active_minutes over 60 unresolved", "silver", "ERROR",
                           _count_where(silver_df, F.col("active_minutes") > 60), silver_rows_for_date, 0,
                           "Hourly active minutes should be capped at 60.", "Check _clean_activity_snapshots active_minutes rule."))

    elif table_name == "health_snapshots":
        bronze_all_bad = _count_where(
            bronze_df,
            F.expr("try_cast(heart_rate AS double)").isNull()
            & F.expr("try_cast(pulse_rate AS double)").isNull()
            & F.expr("try_cast(body_temperature AS double)").isNull()
        )
        silver_all_bad = _count_where(
            silver_df,
            F.col("heart_rate").isNull() & F.col("pulse_rate").isNull() & F.col("body_temperature").isNull()
        )
        rules.append(_rule(run_id, table_name, load_date, "BR_HEALTH_001", "Bronze rows with all vitals unparseable", "bronze", "INFO",
                           bronze_all_bad, bronze_rows, 0,
                           "Rows where all vitals are invalid are not analytically useful.", "Silver dropna should remove rows where all vitals are null."))
        rules.append(_rule(run_id, table_name, load_date, "SV_HEALTH_001", "Silver all-vitals-null rows unresolved", "silver", "ERROR",
                           silver_all_bad, silver_rows_for_date, 0,
                           "Silver should not contain rows where every vital metric is null.", "Check _clean_health_snapshots dropna rule."))

    elif table_name == "feeding_events":
        rules.append(_rule(run_id, table_name, load_date, "BR_FEEDING_001", "Bronze food grams invalid", "bronze", "INFO",
                           _count_where(bronze_df, (F.expr("try_cast(food_dispensed_grams AS double)") <= 0) | (F.expr("try_cast(food_dispensed_grams AS double)") > 1000)), bronze_rows, 0,
                           "Food dispensed should be positive and not an extreme over-dispense.", "Silver should nullify invalid food_dispensed_grams."))
        rules.append(_rule(run_id, table_name, load_date, "SV_FEEDING_001", "Silver food grams invalid unresolved", "silver", "ERROR",
                           _count_where(silver_df, (F.col("food_dispensed_grams") <= 0) | (F.col("food_dispensed_grams") > 1000)), silver_rows_for_date, 0,
                           "Invalid feeding amounts should not survive Silver.", "Check _clean_feeding_events food amount rules."))

    elif table_name == "hydration_events":
        rules.append(_rule(run_id, table_name, load_date, "BR_HYDRATION_001", "Bronze negative water consumption", "bronze", "INFO",
                           _count_where(bronze_df, F.expr("try_cast(water_consumed_ml AS double)") < 0), bronze_rows, 0,
                           "Water consumption should not be negative.", "Silver should ground negative values to 0.0."))
        rules.append(_rule(run_id, table_name, load_date, "SV_HYDRATION_001", "Silver negative water consumption unresolved", "silver", "ERROR",
                           _count_where(silver_df, F.col("water_consumed_ml") < 0), silver_rows_for_date, 0,
                           "Negative hydration values should not survive Silver.", "Check _clean_hydration_events water rule."))

    elif table_name == "device_failures":
        rules.append(_rule(run_id, table_name, load_date, "BR_FAILURE_001", "Bronze severity out of range", "bronze", "INFO",
                           _count_where(bronze_df, (F.expr("try_cast(severity_score AS int)") < 1) | (F.expr("try_cast(severity_score AS int)") > 10)), bronze_rows, 0,
                           "Severity should be between 1 and 10.", "Silver should nullify out-of-range severity_score."))
        rules.append(_rule(run_id, table_name, load_date, "BR_FAILURE_002", "Bronze negative downtime", "bronze", "INFO",
                           _count_where(bronze_df, F.expr("try_cast(downtime_minutes AS int)") < 0), bronze_rows, 0,
                           "Downtime should not be negative.", "Silver should convert negative downtime to absolute value."))
        rules.append(_rule(run_id, table_name, load_date, "SV_FAILURE_001", "Silver severity out of range unresolved", "silver", "ERROR",
                           _count_where(silver_df, (F.col("severity_score") < 1) | (F.col("severity_score") > 10)), silver_rows_for_date, 0,
                           "Invalid severity should not survive Silver.", "Check _clean_device_failures severity rule."))
        rules.append(_rule(run_id, table_name, load_date, "SV_FAILURE_002", "Silver negative downtime unresolved", "silver", "ERROR",
                           _count_where(silver_df, F.col("downtime_minutes") < 0), silver_rows_for_date, 0,
                           "Negative downtime should not survive Silver.", "Check _clean_device_failures downtime rule."))
        rules.append(_rule(run_id, table_name, load_date, "SV_FAILURE_003", "Silver blank failure_type unresolved", "silver", "ERROR",
                           _count_where(silver_df, F.col("failure_type").isNull() | (F.trim(F.col("failure_type")) == "")), silver_rows_for_date, 0,
                           "Blank failure type weakens reliability analysis.", "Check _clean_device_failures Unknown fallback rule."))

    elif table_name == "firmware_deployments":
        rules.append(_rule(run_id, table_name, load_date, "BR_FW_001", "Bronze rollback flag inconsistent", "bronze", "INFO",
                           _count_where(bronze_df, (F.col("rollback_flag").cast("string").isin("true", "True", "TRUE", "1")) & (F.col("deployment_status") != "Rolled Back")), bronze_rows, 0,
                           "Rollback flag should align with deployment status.", "Silver should set rollback_flag false unless status is Rolled Back."))
        rules.append(_rule(run_id, table_name, load_date, "BR_FW_002", "Bronze negative deployment duration", "bronze", "INFO",
                           _count_where(bronze_df, F.expr("try_cast(deployment_duration_seconds AS int)") < 0), bronze_rows, 0,
                           "Deployment duration should not be negative.", "Silver should convert duration to absolute value."))
        rules.append(_rule(run_id, table_name, load_date, "BR_FW_003", "Bronze null firmware_id", "bronze", "INFO",
                           _count_where(bronze_df, F.col("firmware_id").isNull() | (F.trim(F.col("firmware_id")) == "")), bronze_rows, 0,
                           "Firmware deployments must link to firmware.", "Silver should drop orphan deployments."))
        rules.append(_rule(run_id, table_name, load_date, "SV_FW_001", "Silver rollback flag inconsistent unresolved", "silver", "ERROR",
                           _count_where(silver_df, (F.col("rollback_flag") == True) & (F.col("deployment_status") != "Rolled Back")), silver_rows_for_date, 0,
                           "Inconsistent rollback flag should not survive Silver.", "Check _clean_firmware_deployments rollback rule."))
        rules.append(_rule(run_id, table_name, load_date, "SV_FW_002", "Silver negative deployment duration unresolved", "silver", "ERROR",
                           _count_where(silver_df, F.col("deployment_duration_seconds") < 0), silver_rows_for_date, 0,
                           "Negative duration should not survive Silver.", "Check _clean_firmware_deployments duration rule."))
        rules.append(_rule(run_id, table_name, load_date, "SV_FW_003", "Silver null firmware_id unresolved", "silver", "ERROR",
                           _count_where(silver_df, F.col("firmware_id").isNull() | (F.trim(F.col("firmware_id")) == "")), silver_rows_for_date, 0,
                           "Orphan firmware deployments should not survive Silver.", "Check _clean_firmware_deployments dropna rule."))


def profile_fact_table(table_name: str, load_date: str, run_id: str,
                       storage_account: str = "petiot") -> tuple[dict, list[dict]]:
    if table_name not in FACT_TABLES:
        raise ValueError(f"{table_name} is not configured as a fact table.")

    print("=" * 80)
    print(f"[dq_profiler] Profiling fact table: {table_name} | load_date={load_date}")
    print("=" * 80)

    b_path = bronze_path(table_name, storage_account)
    s_path = silver_path(table_name, storage_account)
    bronze_all_df, bronze_error = _try_read_delta(b_path)
    silver_all_df, silver_error = _try_read_delta(s_path)

    bronze_df = filter_bronze_to_load_date(bronze_all_df, load_date) if bronze_all_df is not None else None
    silver_df = filter_silver_to_load_date(silver_all_df, table_name, load_date) if silver_all_df is not None else None

    bronze_rows = bronze_df.count() if bronze_df is not None else 0
    silver_total_rows = silver_all_df.count() if silver_all_df is not None else 0
    silver_rows_for_date = silver_df.count() if silver_df is not None else 0

    rules = []
    _add_general_rules(
        rules, run_id, table_name, load_date, bronze_df, silver_df,
        bronze_rows, silver_rows_for_date, bronze_error, silver_error
    )
    if bronze_df is not None and silver_df is not None:
        _add_table_specific_rules(
            rules, run_id, table_name, load_date, bronze_df, silver_df,
            bronze_rows, silver_rows_for_date
        )

    fail_count = len([r for r in rules if r["status"] in {"FAIL", "ERROR"}])
    warn_count = len([r for r in rules if r["status"] == "WARN"])
    observed_count = len([r for r in rules if r["status"] == "OBSERVED"])

    if fail_count > 0:
        dq_status = "FAIL"
    elif warn_count > 0:
        dq_status = "WARN"
    else:
        dq_status = "PASS"

    row_diff = bronze_rows - silver_rows_for_date
    row_drop_pct = round((row_diff / bronze_rows) * 100, 4) if bronze_rows > 0 else 0.0

    table_summary = {
        "run_id": run_id,
        "table_name": table_name,
        "load_date": normalize_date(load_date),
        "bronze_path": b_path,
        "silver_path": s_path,
        "bronze_rows_for_load_date": int(bronze_rows),
        "silver_total_rows": int(silver_total_rows),
        "silver_rows_for_business_date": int(silver_rows_for_date),
        "bronze_minus_silver_business_date": int(row_diff),
        "row_drop_pct": float(row_drop_pct),
        "dq_status": dq_status,
        "failed_rules": int(fail_count),
        "warning_rules": int(warn_count),
        "observed_bronze_anomaly_rules": int(observed_count),
        "rule_count": int(len(rules)),
        "notes": "Fact-only standalone audit. No existing pipeline notebook was modified.",
    }

    print(f"[dq_profiler] {table_name} status: {dq_status} | failures={fail_count} warnings={warn_count}")
    return table_summary, rules


print("[dq_profiler] Loaded fact table validation rules.")